### Prepare environment

In [ ]:
%run ../../environment/prepare_environment_llm

# Notebook 9.2 - Knowledge Base Preparation

Read workshop markdown files and notebook sources, split them into retrievable chunks, and store the result in Unity Catalog Delta tables.

RAG quality starts with the knowledge base. The model can only cite and reason over the context we retrieve, so this notebook focuses on clean source extraction, stable metadata, and chunk sizes that work well with vector search.

## 1. Configure Target Tables

The workshop stores documents, chunks, and sample questions in Unity Catalog. Keeping these as Delta tables makes the retrieval layer governable and repeatable: the AI Search index is derived from a table, not from ad hoc notebook memory.

In [ ]:
import hashlib
import json
import re
from datetime import datetime, timezone
from pathlib import Path

from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType, TimestampType

dbutils.widgets.text("catalog_name", "ai_ml_in_practice")
dbutils.widgets.text("schema_name", "genai_workshop")
dbutils.widgets.text("repo_root", "")
dbutils.widgets.text("chunk_size", "700")
dbutils.widgets.text("chunk_overlap", "120")

catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
repo_root_widget = dbutils.widgets.get("repo_root").strip()
chunk_size = int(dbutils.widgets.get("chunk_size"))
chunk_overlap = int(dbutils.widgets.get("chunk_overlap"))

documents_table = f"{catalog_name}.{schema_name}.documents"
chunks_table = f"{catalog_name}.{schema_name}.document_chunks"
questions_table = f"{catalog_name}.{schema_name}.rag_questions"

print({
    "catalog_name": catalog_name,
    "schema_name": schema_name,
    "chunk_size": chunk_size,
    "chunk_overlap": chunk_overlap,
})

## 2. Define Parsing and Chunking Helpers

The helper functions below turn repository files into normalized text sections. Markdown files are split by headings, while notebooks contribute authored markdown and code cells. Outputs are intentionally ignored so the knowledge base contains source material, not previous execution noise. Module titles and topic text are parsed from the root README instead of being duplicated in notebook code.

In [ ]:
def find_repo_root() -> Path:
    """Find the repository folder used as the knowledge-base source.

    RAG quality starts with knowing exactly which documents are indexed. The
    widget allows an explicit path when the notebook is imported into a different
    workspace location, while the parent-directory scan keeps local execution
    convenient.
    """
    if repo_root_widget:
        candidate = Path(repo_root_widget).expanduser().resolve()
        if candidate.exists():
            return candidate
        raise FileNotFoundError(f"repo_root does not exist: {candidate}")

    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "9_genai_llms").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root. Set the repo_root widget manually.")


def normalize_text(text: str) -> str:
    """Remove documentation markup that is noisy for retrieval.

    Vector search works best when chunks contain the words a user may ask about.
    Markdown links, image syntax, code fences, and repeated whitespace add tokens
    that rarely help matching, so this function keeps the readable content and
    normalizes it into compact text.
    """
    text = re.sub(r"```.*?```", " ", text, flags=re.DOTALL)
    text = re.sub(r"<br\s*/?>", " ", text)
    text = re.sub(r"!\[[^\]]*\]\([^)]*\)", " ", text)
    text = re.sub(r"\[([^\]]+)\]\([^)]*\)", r"\1", text)
    text = re.sub(r"[#>*_`|]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def load_module_metadata(repo_root: Path) -> dict[str, dict[str, str]]:
    """Read module titles and topic text from the repository README table.

    Retrieval metadata should come from workshop materials, not from a separate
    duplicated mapping in this notebook. The root README already contains the
    module list, goals, and demo topics, so this function extracts those fields
    and turns them into searchable metadata.
    """
    readme_text = (repo_root / "README.md").read_text(encoding="utf-8")
    metadata = {}
    for line in readme_text.splitlines():
        cells = [cell.strip() for cell in line.strip().strip("|").split("|")]
        if len(cells) < 4 or not cells[0].isdigit():
            continue
        module_id, title_cell, goal_cell, topics_cell = cells[:4]
        title = re.sub(r"\[\*\*(.*?)\*\*\]\([^)]*\)", r"\1", title_cell)
        topics = topics_cell.replace("<br>", " ")
        metadata[module_id] = {
            "title": normalize_text(title),
            "retrieval_hint": normalize_text(f"{title} {goal_cell} {topics}"),
        }
    return metadata


def split_sections(text: str) -> list[tuple[str, str]]:
    """Split markdown text into sections using headings as boundaries.

    Section titles become metadata in the chunk table. That metadata is useful
    later because retrieved chunks can tell the answer generator not only which
    file they came from, but also which part of the file they represent.
    """
    sections = []
    current_title = "Overview"
    current_lines = []
    for line in text.splitlines():
        heading = re.match(r"^(#{1,4})\s+(.+)$", line.strip())
        if heading:
            if current_lines:
                sections.append((current_title, "\n".join(current_lines)))
            current_title = heading.group(2).strip()
            current_lines = []
            continue
        current_lines.append(line)
    if current_lines:
        sections.append((current_title, "\n".join(current_lines)))
    return sections


def chunk_text(text: str, size: int, overlap: int) -> list[str]:
    """Break long sections into overlapping retrieval chunks.

    A RAG retriever does not search whole notebooks at once; it searches smaller
    passages. Overlap keeps context from being lost when an important sentence
    sits near a chunk boundary. The boundary logic prefers sentence or word ends
    so chunks remain readable when they are injected into an LLM prompt.
    """
    if len(text) <= size:
        return [text] if text else []

    chunks = []
    start = 0
    while start < len(text):
        end = min(start + size, len(text))
        if end < len(text):
            boundary = max(text.rfind(". ", start, end), text.rfind(" ", start, end))
            if boundary > start + int(size * 0.6):
                end = boundary + 1
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= len(text):
            break
        start = max(0, end - overlap)
    return chunks


def is_workshop_source(relative_path: str) -> bool:
    """Decide which repository files become searchable knowledge.

    Indexing everything would pollute retrieval with Git files, checkpoints, and
    generated metadata. This filter keeps participant-facing markdown and
    notebooks so the chatbot answers from workshop material instead of project
    internals.
    """
    if ".git" in relative_path or ".databricks" in relative_path or ".ipynb_checkpoints" in relative_path:
        return False
    if not relative_path.endswith((".md", ".ipynb")):
        return False
    return bool(re.match(r"^(README\.md|[1-9]_.*|9_genai_llms/.*)", relative_path))


def notebook_sections(path: Path) -> list[tuple[str, str]]:
    """Extract searchable sections from a notebook JSON file.

    Notebooks store cells as JSON arrays. For RAG we want the authored markdown
    and code, but not outputs from previous runs. Code cells are kept as their
    own sections because users often ask how something was implemented.
    """
    notebook = json.loads(path.read_text(encoding="utf-8"))
    sections = []
    code_cell_number = 0
    for cell in notebook.get("cells", []):
        source = cell.get("source", [])
        source_text = "".join(source) if isinstance(source, list) else str(source)
        if not source_text.strip():
            continue
        if cell.get("cell_type") == "markdown":
            sections.extend(split_sections(source_text))
        elif cell.get("cell_type") == "code":
            code_cell_number += 1
            sections.append((f"Code cell {code_cell_number}", source_text))
    return sections


def file_sections(path: Path) -> list[tuple[str, str]]:
    """Return normalized sections for either a markdown file or notebook.

    The downstream chunking code should not care whether a source file was `.md`
    or `.ipynb`. This adapter gives both formats the same `(section_title,
    section_text)` shape.
    """
    if path.suffix == ".ipynb":
        return notebook_sections(path)
    return split_sections(path.read_text(encoding="utf-8"))


def file_title(path: Path, sections: list[tuple[str, str]]) -> str:
    """Choose a human-readable title stored with each document.

    Titles are metadata for inspection, filtering, and citations. The first
    markdown heading is usually the best title; if it is missing, the file name
    is a stable backup title.
    """
    if path.suffix == ".ipynb":
        markdown_titles = [title for title, _ in sections if not title.startswith("Code cell")]
        return markdown_titles[0] if markdown_titles else path.stem
    text = path.read_text(encoding="utf-8")
    return next((line.lstrip("# ").strip() for line in text.splitlines() if line.startswith("#")), path.stem)


repo_root = find_repo_root()
module_metadata_by_id = load_module_metadata(repo_root)
print(f"Repository root: {repo_root}")

## 3. Build Document and Chunk Records

Each source file becomes a document record. Each section becomes one or more chunks with metadata such as module id, module title, source path, and section title. These fields later become citations and filters in RAG.

The table stores both `chunk_text` and `retrieval_text`. `chunk_text` is the clean passage shown to the LLM and used in citations. `retrieval_text` is the search-oriented version: it includes the module title, source path, section title, README-derived module overview, and chunk content. This helps questions such as "Where are classic ML algorithms shown?" match module 4 even when the exact words are split across metadata and notebook content.

In [ ]:
source_files = []
for path in sorted(repo_root.rglob("*")):
    relative_path = path.relative_to(repo_root).as_posix()
    if is_workshop_source(relative_path):
        source_files.append(path)

documents = []
chunks = []
loaded_at = datetime.now(timezone.utc)

for path in source_files:
    relative_path = path.relative_to(repo_root).as_posix()
    module_match = re.match(r"^(\d+)_", relative_path)
    module_id = module_match.group(1) if module_match else "0"
    module_metadata = module_metadata_by_id.get(module_id, {})
    sections = file_sections(path)
    first_heading = file_title(path, sections)
    module_title = module_metadata.get("title", first_heading)
    module_retrieval_hint = module_metadata.get("retrieval_hint", "")
    document_id = hashlib.sha256(relative_path.encode("utf-8")).hexdigest()[:16]

    documents.append({
        "document_id": document_id,
        "module_id": module_id,
        "module_title": module_title,
        "source_path": relative_path,
        "loaded_at": loaded_at,
    })

    for section_title, section_text in sections:
        clean_text = normalize_text(section_text)
        for chunk_number, chunk in enumerate(chunk_text(clean_text, chunk_size, chunk_overlap), start=1):
            chunk_key = f"{relative_path}:{section_title}:{chunk_number}:{chunk}"
            retrieval_text = "\n".join([
                f"Module {module_id}: {module_title}",
                f"Source path: {relative_path}",
                f"Section: {section_title}",
                f"Module overview: {module_retrieval_hint}",
                f"Content: {chunk}",
            ])
            chunks.append({
                "chunk_id": hashlib.sha256(chunk_key.encode("utf-8")).hexdigest(),
                "document_id": document_id,
                "module_id": module_id,
                "module_title": module_title,
                "source_path": relative_path,
                "section_title": section_title,
                "chunk_text": chunk,
                "retrieval_text": retrieval_text,
                "updated_at": loaded_at,
            })

print({"documents": len(documents), "chunks": len(chunks)})

## 4. Write Delta Tables

The chunks table is the source for AI Search. Change Data Feed is enabled because Delta Sync indexes use table changes to keep search data up to date.

The AI Search index in the next notebook should embed `retrieval_text`, while prompts should still cite and display `chunk_text`.

In [ ]:

documents_schema = StructType([
    StructField("document_id", StringType(), False),
    StructField("module_id", StringType(), False),
    StructField("module_title", StringType(), False),
    StructField("source_path", StringType(), False),
    StructField("loaded_at", TimestampType(), False),
])

chunks_schema = StructType([
    StructField("chunk_id", StringType(), False),
    StructField("document_id", StringType(), False),
    StructField("module_id", StringType(), False),
    StructField("module_title", StringType(), False),
    StructField("source_path", StringType(), False),
    StructField("section_title", StringType(), False),
    StructField("chunk_text", StringType(), False),
    StructField("retrieval_text", StringType(), False),
    StructField("updated_at", TimestampType(), False),
])

documents_df = spark.createDataFrame(documents, schema=documents_schema)
chunks_df = spark.createDataFrame(chunks, schema=chunks_schema)

documents_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(documents_table)
chunks_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(chunks_table)
spark.sql(f"ALTER TABLE {chunks_table} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

sample_questions = [
    ("q1", "How do ML pipelines differ from MLOps in this workshop?"),
    ("q2", "What does RAG add to a language model application?"),
    ("q3", "Which module explains feature engineering?"),
]
questions_df = spark.createDataFrame(sample_questions, ["question_id", "question"])
questions_df = questions_df.withColumn("created_at", F.current_timestamp())
questions_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(questions_table)

display(spark.table(chunks_table).orderBy("module_id", "source_path", "section_title").limit(20))

## 5. Inspect Chunk Distribution

This summary confirms that the corpus covers multiple workshop modules. A healthy RAG index should not contain only one document or one module unless the use case is intentionally narrow.

In [ ]:
display(
    spark.table(chunks_table)
    .groupBy("module_id", "module_title")
    .count()
    .orderBy("module_id")
)